In [1]:
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os

load_dotenv()
youtube = build("youtube", "v3", developerKey=os.getenv("YOUTUBE_API_KEY"))

req = youtube.search().list(
    q="AI generated music",
    part="snippet",
    maxResults=3
)
res = req.execute()
print(res["items"][0]["snippet"]["title"])

e:\ai-music-sentiment\.venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


AI-Generated Music is Wild!


In [ ]:
# src/collect_youtube.py

import os
import time
import csv
from datetime import datetime
from googleapiclient.discovery import build
from dotenv import load_dotenv

load_dotenv()

# 수집할 영상 ID
VIDEO_IDS = [
    "H3_E37znw18",
    "hMV3pH9rdHw",
    "ECLy6JnBdoY",
    "vA63-nDMYGg",
]

MAX_COMMENTS_PER_VIDEO = 500  # 영상당 최대 댓글 수
OUTPUT_DIR = "../data/raw/youtube"


def get_youtube_client():
    return build("youtube", "v3", developerKey=os.getenv("YOUTUBE_API_KEY"))


def fetch_comments(youtube, video_id, max_comments=500):
    comments = []
    next_page_token = None

    while len(comments) < max_comments:
        try:
            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, max_comments - len(comments)),
                pageToken=next_page_token,
                textFormat="plainText",
                order="relevance"
            ).execute()

        except Exception as e:
            print(f"  [!] 에러 발생 (video: {video_id}): {e}")
            break

        for item in response.get("items", []):
            snippet = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "video_id":      video_id,
                "comment_id":    item["snippet"]["topLevelComment"]["id"],
                "author":        snippet.get("authorDisplayName", ""),
                "text":          snippet.get("textDisplay", ""),
                "likes":         snippet.get("likeCount", 0),
                "reply_count":   item["snippet"].get("totalReplyCount", 0),
                "published_at":  snippet.get("publishedAt", ""),
                "collected_at":  datetime.utcnow().isoformat(),
                "platform":      "youtube",
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.5)  # 페이지 간 딜레이

    return comments


def save_to_csv(comments, video_id):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    filename = f"{OUTPUT_DIR}/yt_{video_id}_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"

    fieldnames = [
        "video_id", "comment_id", "author", "text",
        "likes", "reply_count", "published_at", "collected_at", "platform"
    ]

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(comments)

    return filename


def main():
    youtube = get_youtube_client()
    total = 0

    print(f"수집 시작 — 영상 {len(VIDEO_IDS)}개\n")

    for i, video_id in enumerate(VIDEO_IDS, 1):
        print(f"[{i}/{len(VIDEO_IDS)}] video_id: {video_id}")

        comments = fetch_comments(youtube, video_id, MAX_COMMENTS_PER_VIDEO)

        if comments:
            path = save_to_csv(comments, video_id)
            total += len(comments)
            print(f"  → {len(comments)}개 수집 완료 → {path}")
        else:
            print(f"  → 댓글 없음 또는 비활성화된 영상")

        time.sleep(1.0)  # 영상 간 딜레이

    print(f"\n완료 — 총 {total}개 댓글 수집")


if __name__ == "__main__":
    main()

수집 시작 — 영상 3개

[1/3] video_id: H3_E37znw18
  → 119개 수집 완료 → ../data/raw/youtube/yt_H3_E37znw18_20260328_014545.csv
[2/3] video_id: hMV3pH9rdHw
  → 346개 수집 완료 → ../data/raw/youtube/yt_hMV3pH9rdHw_20260328_014549.csv
[3/3] video_id: ECLy6JnBdoY
  → 500개 수집 완료 → ../data/raw/youtube/yt_ECLy6JnBdoY_20260328_014554.csv

완료 — 총 965개 댓글 수집


In [ ]:
# src/collect_youtube.py

import os
import time
import csv
from datetime import datetime
from googleapiclient.discovery import build
from dotenv import load_dotenv

load_dotenv()

# ✏️ 여기에 수집할 영상 ID 추가
VIDEO_IDS = [
    "H3_E37znw18",
    "hMV3pH9rdHw",
    "ECLy6JnBdoY",
    "vA63-nDMYGg",
    "4He-MET8fik",
    "ZZ0BOEOtD2U",
    "Ey75Xw_ikqs",
    "jHrkQ928VNI",
    "2EKzWNcXrM0",
    "wBf6Yfahb2I",
    "a_qhYjVyawk",
    "QjHRZcFD6q4",
    "Z_2g0L3Hzgw",
    "1aZTpqKiH-M",
    "QtZDkgzjmQI",
    "v81xlIMSF-I"
]

MAX_COMMENTS_PER_VIDEO = 500
OUTPUT_DIR = "../data/raw/youtube"


def get_youtube_client():
    return build("youtube", "v3", developerKey=os.getenv("YOUTUBE_API_KEY"))


def already_collected(video_id):
    """이미 수집된 영상 ID인지 확인"""
    if not os.path.exists(OUTPUT_DIR):
        return False
    existing = os.listdir(OUTPUT_DIR)
    return any(f.startswith(f"yt_{video_id}_") for f in existing)


def fetch_comments(youtube, video_id, max_comments=500):
    comments = []
    next_page_token = None

    while len(comments) < max_comments:
        try:
            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, max_comments - len(comments)),
                pageToken=next_page_token,
                textFormat="plainText",
                order="relevance"
            ).execute()

        except Exception as e:
            print(f"  [!] 에러 발생 (video: {video_id}): {e}")
            break

        for item in response.get("items", []):
            snippet = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "video_id":     video_id,
                "comment_id":   item["snippet"]["topLevelComment"]["id"],
                "author":       snippet.get("authorDisplayName", ""),
                "text":         snippet.get("textDisplay", ""),
                "likes":        snippet.get("likeCount", 0),
                "reply_count":  item["snippet"].get("totalReplyCount", 0),
                "published_at": snippet.get("publishedAt", ""),
                "collected_at": datetime.utcnow().isoformat(),
                "platform":     "youtube",
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.5)

    return comments


def save_to_csv(comments, video_id):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    filename = f"{OUTPUT_DIR}/yt_{video_id}_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"

    fieldnames = [
        "video_id", "comment_id", "author", "text",
        "likes", "reply_count", "published_at", "collected_at", "platform"
    ]

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(comments)

    return filename


def main():
    youtube = get_youtube_client()
    total = 0

    print(f"수집 시작 — 영상 {len(VIDEO_IDS)}개\n")

    for i, video_id in enumerate(VIDEO_IDS, 1):
        print(f"[{i}/{len(VIDEO_IDS)}] video_id: {video_id}")

        # 중복 확인
        if already_collected(video_id):
            print(f"  → 이미 수집된 영상, 스킵\n")
            continue

        comments = fetch_comments(youtube, video_id, MAX_COMMENTS_PER_VIDEO)

        if comments:
            path = save_to_csv(comments, video_id)
            total += len(comments)
            print(f"  → {len(comments)}개 수집 완료 → {path}\n")
        else:
            print(f"  → 댓글 없음 또는 비활성화된 영상\n")

        time.sleep(1.0)

    print(f"완료 — 총 {total}개 댓글 수집")


if __name__ == "__main__":
    main()

수집 시작 — 영상 15개

[1/15] video_id: H3_E37znw18
  → 이미 수집된 영상, 스킵

[2/15] video_id: hMV3pH9rdHw
  → 이미 수집된 영상, 스킵

[3/15] video_id: ECLy6JnBdoY
  → 이미 수집된 영상, 스킵

[4/15] video_id: vA63-nDMYGg
  → 이미 수집된 영상, 스킵

[5/15] video_id: 4He-MET8fik
  → 이미 수집된 영상, 스킵

[6/15] video_id: ZZ0BOEOtD2U
  → 이미 수집된 영상, 스킵

[7/15] video_id: Ey75Xw_ikqs
  → 이미 수집된 영상, 스킵

[8/15] video_id: jHrkQ928VNI
  → 이미 수집된 영상, 스킵

[9/15] video_id: 2EKzWNcXrM0
  → 이미 수집된 영상, 스킵

[10/15] video_id: wBf6Yfahb2I
  → 이미 수집된 영상, 스킵

[11/15] video_id: a_qhYjVyawk
  → 이미 수집된 영상, 스킵

[12/15] video_id: QjHRZcFD6q4
  → 이미 수집된 영상, 스킵

[13/15] video_id: Z_2g0L3Hzgw
  → 이미 수집된 영상, 스킵

[14/15] video_id: 1aZTpqKiH-M
  → 이미 수집된 영상, 스킵

[15/15] video_id: QtZDkgzjmQI
  → 500개 수집 완료 → ../data/raw/youtube/yt_QtZDkgzjmQI_20260328_023703.csv

완료 — 총 500개 댓글 수집
